In [6]:
import sqlite3
import os 

os.makedirs("data/databases",exist_ok=True)


# Sample DB
conn=sqlite3.connect("data/databases/company.db")
cursor=conn.cursor()

# Creating a Table
conn.execute('''Create table if not exists Employee
                (id int Primary key , name text , status text , job_role text, salary real )''')


conn.execute('''Create table if not exists Projects
                (id int Primary Key , name text , status text , budget real , lead_id int)''')

In [7]:
Employee=[
    (1, "John", "Active", "Data Analytics", 95000),
    (2, "Kyle", "Active", "ML Engineer", 990000),
    (3, "Kaleb", "Non-Active", "Front-End Developer", 900000),
    (4, "Krista", "Active", "HR", 500000),
    (5 , "Eliot", "Active", "Cyber Security", 100000 )
]

Projects=[
    (1,'Rag Implementation',"Active",150000,1),
    (2,"Data Pipeline","Completed", 800000,2),
    (3,"Customer Portal","Planning",200000,3),
    (4,"Ml Platform", "Active", 2500000,4),
    (5,"Cloud Services", "Ative",1200000,5)
]

In [8]:
cursor.executemany('Insert or Replace into Employee Values (?,?,?,?,?)',Employee)
cursor.executemany('Insert or Replace into Projects Values (?,?,?,?,?)',Projects)

In [9]:
cursor.execute("Select * from Employee")

In [10]:
conn.commit()
conn.close()

In [11]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

db=SQLDatabase.from_uri("sqlite:///data/databases/company.db")

print(f"Tables: {db.get_usable_table_names()}")
print(f"\nTable DDL:")
print(db.get_table_info())

C:\Users\Admin\AppData\Local\Temp\ipykernel_7128\1377329553.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


Tables: ['Employee', 'Projects']

Table DDL:

CREATE TABLE "Employee" (
	id INTEGER, 
	name TEXT, 
	status TEXT, 
	job_role TEXT, 
	salary REAL, 
	PRIMARY KEY (id)
)

/*
3 rows from Employee table:
id	name	status	job_role	salary
1	John	Active	Data Analytics	95000.0
2	Kyle	Active	ML Engineer	990000.0
3	Kaleb	Non-Active	Front-End Developer	900000.0
*/


CREATE TABLE "Projects" (
	id INTEGER, 
	name TEXT, 
	status TEXT, 
	budget REAL, 
	lead_id INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from Projects table:
id	name	status	budget	lead_id
1	Rag Implementation	Active	150000.0	1
2	Data Pipeline	Completed	800000.0	2
3	Customer Portal	Planning	200000.0	3
*/


In [13]:
from typing import List
from langchain_core.documents import Document
import sqlite3

def sql_to_docs(db_path:str)->List[Document]:
    conn=sqlite3.connect(db_path)
    cursor=conn.cursor()
    documents=[]

    cursor.execute("Select name from sqlite_master where type='table';")
    tables=cursor.fetchall()

    for table in tables:
        table_name=table[0]

        cursor.execute(f"pragma table_info({table_name});")
        columns=cursor.fetchall()
        column_name=[col[1] for col in columns]
        cursor.execute(f"Select * from {table_name}")
        rows=cursor.fetchall()


        table_content=f"Table: {table_name}\n"
        table_content+=f"Columns: {','.join(column_name)}\n"
        table_content+=f"Total Records: {len(rows)}\n\n"
        table_content+="Sample Records: \n"

        for row in rows[:5]:
            record=dict(zip(column_name,row))
            table_content+=f"{record}\n"

        doc=Document(
            page_content=table_content,
            metadata={
                'source':db_path,
                'table_name':table_name,
                'num_records':len(rows),
                'data_type':'sql_table'
            }
        )
        documents.append(doc)
    return documents


sql_to_docs("data/databases/company.db")

[Document(metadata={'source': 'data/databases/company.db', 'table_name': 'Employee', 'num_records': 5, 'data_type': 'sql_table'}, page_content="Table: Employee\nColumns: id,name,status,job_role,salary\nTotal Records: 5\n\nSample Records: \n{'id': 1, 'name': 'John', 'status': 'Active', 'job_role': 'Data Analytics', 'salary': 95000.0}\n{'id': 2, 'name': 'Kyle', 'status': 'Active', 'job_role': 'ML Engineer', 'salary': 990000.0}\n{'id': 3, 'name': 'Kaleb', 'status': 'Non-Active', 'job_role': 'Front-End Developer', 'salary': 900000.0}\n{'id': 4, 'name': 'Krista', 'status': 'Active', 'job_role': 'HR', 'salary': 500000.0}\n{'id': 5, 'name': 'Eliot', 'status': 'Active', 'job_role': 'Cyber Security', 'salary': 100000.0}\n"),
 Document(metadata={'source': 'data/databases/company.db', 'table_name': 'Projects', 'num_records': 5, 'data_type': 'sql_table'}, page_content="Table: Projects\nColumns: id,name,status,budget,lead_id\nTotal Records: 5\n\nSample Records: \n{'id': 1, 'name': 'Rag Implementati